# Data Loading

In [19]:
import pandas as pd
import re
#import nltk

import spacy

# Import VADER model
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
sia = SentimentIntensityAnalyzer()

# Import Doc for extending the SpaCy pipeline
from spacy.tokens import Doc

In [20]:
games_df = pd.read_csv('games.csv')
reviews_df = pd.read_csv('dataset.csv')
#reviews_df.head()

games_df.head()
#reviews_df.head()

,AppID,Name,Release date,Estimated owners,Peak CCU,Required age,Price,DiscountDLC count,About the game,Supported languages,...,Average playtime two weeks,Median playtime forever,Median playtime two weeks,Developers,Publishers,Categories,Genres,Tags,Screenshots,Movies
2539430,Black Dragon Mage Playtest,"Aug 1, 2023",0 - 0,0,0,0.00,0,0,NaN,[],...,0,0,0,NaN,NaN,NaN,NaN,NaN,https://shared.akamai.steamstatic.com/store_it...,NaN
496350,Supipara - Chapter 1 Spring Has Come!,"Jul 29, 2016",0 - 20000,0,0,5.24,65,0,"Springtime, April: when the cherry trees come ...",['English'],...,0,8,0,minori,MangaGamer,"Single-player,Steam Trading Cards,Steam Cloud,...",Adventure,"Adventure,Visual Novel,Anime,Cute",https://shared.akamai.steamstatic.com/store_it...,NaN
1034400,Mystery Solitaire The Black Raven,"May 6, 2019",0 - 20000,0,0,4.99,0,0,"Immerse yourself in the most beloved, mystical...","['English', 'French', 'German', 'Russian']",...,0,0,0,Somer Games,8floor,"Single-player,Family Sharing",Casual,"Casual,Card Game,Solitaire,Puzzle,Hidden Objec...",https://shared.akamai.steamstatic.com/store_it...,NaN
3292190,버튜버 파라노이아 - Vtuber Paranoia,"Oct 31, 2024",0 - 20000,1,0,8.99,0,1,"synopsis 'Hello, I'm Hiyoro, a new YouTuber!' ...",['Korean'],...,0,0,0,유진게임즈,유진게임즈,"Single-player,Steam Achievements,Family Sharing","Casual,Indie,Simulation",NaN,https://shared.akamai.steamstatic.com/store_it...,NaN
3631080,Maze Quest VR,"Apr 24, 2025",0 - 20000,0,0,4.99,0,0,Its not just a Maze; its a Quest! Enter the ca...,['English'],...,0,0,0,Reality Expanded LLC,Reality Expanded LLC,"Single-player,VR Only,Steam Leaderboards,Famil...","Action,Early Access",NaN,https://shared.akamai.steamstatic.com/store_it...,NaN


In [21]:
listed_genres = set()
for genres in games_df['Genres'].unique().tolist():
    for genre in str(genres).split(','):
        listed_genres.add(genre)


listed_tags = set()
for tags in games_df['Tags'].unique().tolist():
    for tag in str(tags).split(','):
        listed_tags.add(tag)

print(listed_genres)
print(len(listed_genres))

print(listed_tags)
print(len(listed_tags))

{'Casual', 'Violent', 'Simulation', 'Early Access', 'Sports', 'Racing', 'Education', 'Accounting', 'Sexual Content', 'Web Publishing', '360 Video', 'RPG', 'Indie', 'Design & Illustration', 'Action', 'Massively Multiplayer', 'Gore', 'Video Production', 'Episodic', 'Audio Production', 'Free To Play', 'Software Training', 'Strategy', 'Game Development', 'Nudity', 'Documentary', 'Animation & Modeling', 'Tutorial', 'Short', 'Photo Editing', 'Utilities', 'Movie', 'nan', 'Adventure'}
34
{'Fighting', 'Class-Based', 'Third Person', 'Early Access', 'Otome', 'Parody', 'Resource Management', 'Masterpiece', 'Lore-Rich', 'World War I', 'Jump Scare', 'Jet', 'Transhumanism', 'Science', 'Snow', 'Cyberpunk', 'Video Production', 'Episodic', 'Dating Sim', 'Abstract', 'Dungeon Crawler', 'Space Sim', 'Top-Down', 'Political', 'Assassin', 'Robots', 'Vampire', 'MMORPG', 'Gun Customization', 'Typing', 'Choices Matter', 'Relaxing', 'e-sports', 'Violent', 'Colorful', 'Tactical', 'Action RTS', 'Naval', 'First-Pers

In [22]:
narrative_games = games_df[games_df['Tags'].str.contains('Story Rich') | games_df['Tags'].str.contains('Narrative') | games_df['Tags'].str.contains('Choices Matter')]

narrative_games.head()

,AppID,Name,Release date,Estimated owners,Peak CCU,Required age,Price,DiscountDLC count,About the game,Supported languages,...,Average playtime two weeks,Median playtime forever,Median playtime two weeks,Developers,Publishers,Categories,Genres,Tags,Screenshots,Movies
1108640,OMNIMUS,"Sep 25, 2019",0 - 20000,0,0,0.99,0,1,Omnimus is an interactive visual novel in an u...,"['English', 'Russian']",...,0,0,0,Angry games,Total,"Single-player,Family Sharing","Adventure,Indie","Adventure,Indie,Visual Novel,Interactive Ficti...",https://shared.akamai.steamstatic.com/store_it...,NaN
1514380,COGEN: Sword of Rewind / COGEN: 大鳥こはくと刻の剣,"Jan 26, 2022",20000 - 50000,0,0,12.49,0,3,■Defy Death by Rewinding Time! Any hit you tak...,"['English', 'Simplified Chinese', 'Traditional...",...,0,0,0,"Gemdrops, Inc.","Gemdrops, Inc.","Single-player,Steam Achievements,Full controll...",Action,"Action-Adventure,Pixel Graphics,Time Manipulat...",https://shared.akamai.steamstatic.com/store_it...,NaN
1479370,Pixel Game Maker Series Werewolf Princess Kaguya,"Dec 20, 2020",0 - 20000,0,0,9.99,0,0,A stylish &amp; exciting action game! Journey ...,"['English', 'Simplified Chinese', 'Traditional...",...,0,0,0,SERIALGAMES Inc.,Gotcha Gotcha Games,"Single-player,Full controller support,Family S...","Action,Indie","Action,Arcade,Character Action Game,2D Platfor...",https://shared.akamai.steamstatic.com/store_it...,NaN
3098390,Spookstopia: Atomic Division,"Jul 31, 2024",0 - 20000,0,0,2.99,0,0,What is possible between the dimensions of Hel...,"['English', 'Russian']",...,0,0,0,TexFrendza,TexFrendza,"Single-player,Family Sharing","Indie,Early Access","Adventure,Platformer,2D Platformer,2D,Mystery,...",https://shared.akamai.steamstatic.com/store_it...,NaN
3215680,Who is Paul,"Oct 21, 2024",0 - 20000,0,0,0.89,70,0,WARNING: The game's content includes sudden lo...,"['English', 'French', 'Italian', 'German', 'Sp...",...,0,0,0,Atlas Sf,Atlas Sf,"Single-player,Steam Achievements,Family Sharing","Action,Adventure,Indie","Action,Adventure,FPS,Dark,Story Rich,Stealth,A...",https://shared.akamai.steamstatic.com/store_it...,NaN


In [23]:
narrative_game_names = narrative_games["AppID"].unique().tolist()

filtered_reviews = reviews_df[reviews_df["app_name"].isin(narrative_game_names)]

print(len(filtered_reviews))
print(len(filtered_reviews["app_name"].unique().tolist()))

filtered_reviews.head()


1236465
1101


,app_id,app_name,review_text,review_score,review_votes
46177,104600,Portal 2 - The Final Hours,"First of all, this is no game but a virtual in...",1,1
46178,104600,Portal 2 - The Final Hours,The Final Hours is not a game (though can't yo...,1,1
46179,104600,Portal 2 - The Final Hours,Get this if you are a geek about how games are...,1,1
46180,104600,Portal 2 - The Final Hours,IT'S NOT A GAME! But it's a very interesting r...,1,1
46181,104600,Portal 2 - The Final Hours,Everything you would want to know about Portal...,1,1


In [24]:
# Cleaning up narrative games by columns that are relevant to us
narrative_games_cleaned = narrative_games[['AppID', 'Name', 'Estimated owners', 'Price', 'About the game', 'Positive', 'Negative', 'Score rank', 'Recommendations', 'Categories', 'Genres', 'Tags']]

available_names = filtered_reviews["app_name"].unique()
narrative_games_available = narrative_games_cleaned[narrative_games_cleaned["AppID"].isin(available_names)]

review_count = []
pos_ratio = []

for i in range (len(narrative_games_available)):
    game = narrative_games_available.iloc[i]
    game_reviews = filtered_reviews[filtered_reviews['app_name']==game.get('AppID')]
    game_review_count = len(game_reviews)
    game_postiive_reviews = sum(1 for review in game_reviews['review_score'] if review >0)
    game_pos_ratio = game_postiive_reviews/game_review_count
    review_count.append(game_review_count)
    pos_ratio.append(game_pos_ratio)


narrative_games_available['review_count'] = review_count
narrative_games_available['pos_ratio'] = pos_ratio

narrative_games_available.head()

,AppID,Name,Estimated owners,Price,About the game,Positive,Negative,Score rank,Recommendations,Categories,Genres,Tags,review_count,pos_ratio
558660,it's always monday,"Nov 22, 2016",0,0,it's always monday- is a first person explorat...,228,61,NaN,0,"Single-player,Family Sharing","Adventure,Indie,Simulation","Adventure,Indie,Simulation,First-Person,Psycho...",77,0.714286
242700,Injustice: Gods Among Us Ultimate Edition,"Nov 12, 2013",69,0,Injustice: Gods Among Us Ultimate Edition enha...,16534,2651,NaN,9627,"Single-player,Multi-player,Steam Achievements,...",Action,"Fighting,Superhero,Action,Multiplayer,Comic Bo...",2966,0.786244
277470,The Book of Legends,"Mar 14, 2014",0,0,"The Demon of Fear, Azutura, is slowly but sure...",191,94,NaN,139,"Single-player,Steam Achievements,Full controll...","Adventure,Indie,RPG","RPG,RPGMaker,Indie,Adventure,Anime,JRPG,Single...",135,0.674074
222480,Resident Evil Revelations,"May 20, 2013",101,75,Resident Evil ® Revelations returns redefined ...,12258,2947,NaN,10007,"Single-player,Multi-player,Co-op,Steam Achieve...","Action,Adventure","Horror,Action,Zombies,Survival Horror,Third-Pe...",1519,0.813035
311010,The Way,"May 20, 2016",0,94,Journey to another planet and discover its sec...,1053,212,NaN,883,"Single-player,Steam Achievements,Steam Trading...","Action,Adventure,Indie","Adventure,Indie,Action,Pixel Graphics,Platform...",167,0.904192


## Data Cleaning

In [25]:
#filters for ascii characters
filter_ascii = filtered_reviews["review_text"].replace(to_replace=r'[^\x00-\x7f]', value='', regex=True)

#removes new lines
remove_newline = filter_ascii.replace(to_replace=r'\d', value='', regex=True)

#removes certain characters used in reviews
remove_otherchars = remove_newline.replace(to_replace=r'[,()*/"1234567890+|]', value='', regex=True)

#removes punctuations
replace_punctuations = remove_otherchars.replace(to_replace=r'[.!?]+', value = ' | ', regex=True)

#remove dashes not used to hyphenate words
remove_nonhyphen_dashes = replace_punctuations.replace(to_replace=r'- | -', value=' ', regex=True)

#remove extra whitespace
remove_extra_whitespace = remove_nonhyphen_dashes.replace(to_replace=r' +', value=' ', regex=True)

# assign cleaned entries back to the data frame
new = remove_extra_whitespace
filtered_reviews['review_text'] = new

# apply lowercasing
filtered_reviews['review_text'] = filtered_reviews['review_text'].str.lower()
filtered_reviews

print(filtered_reviews)

         app_id                    app_name  \
46177    104600  Portal 2 - The Final Hours   
46178    104600  Portal 2 - The Final Hours   
46179    104600  Portal 2 - The Final Hours   
46180    104600  Portal 2 - The Final Hours   
46181    104600  Portal 2 - The Final Hours   
...         ...                         ...   
6404109    9880            Champions Online   
6404110    9880            Champions Online   
6404111    9880            Champions Online   
6404112    9880            Champions Online   
6404113    9880            Champions Online   

                                               review_text  review_score  \
46177    first of all this is no game but a virtual int...             1   
46178    the final hours is not a game though can't you...             1   
46179    get this if you are a geek about how games are...             1   
46180    it's not a game | but it's a very interesting ...             1   
46181    everything you would want to know about portal

In [26]:
## additional cleaning based on Kaggle User Ali Marjani, PhD
## via https://www.kaggle.com/code/dardodel/steam-reviews-auto-topic-modeling-w-transformers/notebook

print("before removing duplicates: " + str(filtered_reviews.shape))

# drop duplicates
## NOTE: only this actually decreased stuff lmao
filtered_reviews.drop_duplicates(['review_text', 'review_score'], inplace = True)
print("after removing duplicates: " + str(filtered_reviews.shape))


before removing duplicates: (1236465, 5)
after removing duplicates: (1017234, 5)


In [ ]:
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()

word_tokens = [ (str(review)).split() for review in filtered_reviews['review_text']]

stop_words = set(stopwords.words('english'))
stop_words.add('us')

lemmatized_reviews = []
for tokens in word_tokens:
    lemmatized_reviews.append([lemmatizer.lemmatize(token) for token in tokens if token not in stop_words or token=='not'])

lemmatized_reviews = [" ".join(review) for review in lemmatized_reviews]

filtered_reviews["lemmatized_reviews"] = lemmatized_reviews
filtered_reviews.head()

,app_id,app_name,review_text,review_score,review_votes,lemmatized_reviews
46177,104600,Portal 2 - The Final Hours,first of all this is no game but a virtual int...,1,1,first game virtual interactive book good one m...
46178,104600,Portal 2 - The Final Hours,the final hours is not a game though can't you...,1,1,final hour not game though can't imagine one s...
46179,104600,Portal 2 - The Final Hours,get this if you are a geek about how games are...,1,1,get geek game made |
46180,104600,Portal 2 - The Final Hours,it's not a game | but it's a very interesting ...,1,1,not game | interesting reading interested crea...
46181,104600,Portal 2 - The Final Hours,everything you would want to know about portal...,1,1,everything would want know portal right intera...
...,...,...,...,...,...,...
6404107,9880,Champions Online,i paid for a month subscription to check out a...,-1,1,paid month subscription check much full game c...
6404108,9880,Champions Online,my absolute favorite superhero games even thou...,-1,0,absolute favorite superhero game even though g...
6404109,9880,Champions Online,the gane starts to get boreing when you reatch...,-1,0,gane start get boreing reatch level know every...
6404110,9880,Champions Online,best character creator i've seen | edit: they ...,-1,0,best character creator seen | edit: removed it...


In [28]:
# Getting highest rating based on number of reviews and number of positive reviews
top5_df = narrative_games_available.nlargest(6, ["review_count", "pos_ratio"], 'first')
top5_games = []

for i in range (len(top5_df)):
    game = top5_df.iloc[i]
    top5_games.append(game.get("AppID")) #Get all the game names
    
#weird thing of data where Portal 2 appears twice
top5_df = top5_df.drop(top5_df.index[1])

top5_df

,AppID,Name,Estimated owners,Price,About the game,Positive,Negative,Score rank,Recommendations,Categories,Genres,Tags,review_count,pos_ratio
391540,Undertale,"Sep 15, 2015",3331,75,"Welcome to UNDERTALE. In this RPG, you control...",264082,8850,NaN,276713,"Single-player,Steam Trading Cards,Remote Play ...","Indie,RPG","Great Soundtrack,Story Rich,Choices Matter,Mul...",51918,0.960187
620,Portal 2,"Apr 18, 2011",1959,80,Portal 2 draws from the award-winning formula ...,427835,5675,NaN,374061,"Single-player,Multi-player,Co-op,Online Co-op,...","Action,Adventure","Platformer,Puzzle,First-Person,Dark Humor,Stor...",38924,0.987771
22380,Fallout: New Vegas,"Oct 19, 2010",3892,50,Welcome to Vegas. New Vegas. It’s the kind of ...,218763,8937,NaN,190587,"Single-player,Steam Achievements,Partial Contr...","Action,RPG","Open World,RPG,Post-apocalyptic,Singleplayer,M...",32963,0.956345
292030,The Witcher 3: Wild Hunt,"May 18, 2015",15893,90,NaN,802993,32356,NaN,797928,"Single-player,Steam Achievements,Steam Trading...",RPG,"Open World,RPG,Story Rich,Atmospheric,Mature,F...",31853,0.968355
8870,BioShock Infinite,"Mar 25, 2013",301,75,"Indebted to the wrong people, with his life on...",131970,8712,NaN,107475,"Single-player,Steam Achievements,Full controll...",Action,"FPS,Story Rich,Action,Singleplayer,Steampunk,A...",31140,0.941458


In [29]:
# Did it this way since games with no reviews will register as 0 and will clog up the space for lowest rated games
# We still want a significant amount of reviews to base what is actually bad vs what is underrepresented/underreviewed
cond = narrative_games_available.get("review_count") >= 1000
temp_bot_df = narrative_games_available.where(cond)
bot5_df = temp_bot_df.nsmallest(5, "pos_ratio", 'first')

bot5_games = []

for i in range (len(bot5_df)):
    game = bot5_df.iloc[i]
    bot5_games.append(game.get("AppID")) #Get all the game names

bot5_df.head()
#print(bot5_games)

,AppID,Name,Estimated owners,Price,About the game,Positive,Negative,Score rank,Recommendations,Categories,Genres,Tags,review_count,pos_ratio
360430,Mafia III: Definitive Edition,"May 19, 2020",869.0,80.0,Mafia III: Definitive Edition includes the mai...,32366.0,24142.0,NaN,38656.0,"Single-player,Steam Achievements,Full controll...","Action,Adventure","Crime,Open World,Action,Story Rich,Singleplaye...",1279.0,0.057076
377160,Fallout 4,"Nov 9, 2015",11179.0,60.0,"From Bethesda Game Studios, the award-winning ...",326054.0,66233.0,NaN,276492.0,"Single-player,Steam Achievements,Full controll...",RPG,"Open World,Post-apocalyptic,Singleplayer,Explo...",6121.0,0.341121
72850,The Elder Scrolls V: Skyrim,"Nov 10, 2011",1463.0,0.0,EPIC FANTASY REBORN The next chapter in the hi...,300915.0,16364.0,NaN,181105.0,"Single-player,Steam Achievements,Steam Trading...",RPG,"Open World,RPG,Fantasy,Adventure,Dragons,Modda...",2260.0,0.356637
32500,STAR WARS™: The Force Unleashed™ II,"Oct 26, 2010",43.0,50.0,The Star Wars® Saga continues with Star Wars®:...,4692.0,2696.0,NaN,5544.0,"Single-player,Partial Controller Support,Steam...",Action,"Action,Singleplayer,Hack and Slash,Third Perso...",1274.0,0.512559
208650,Batman™: Arkham Knight,"Jun 23, 2015",1587.0,80.0,Batman™: Arkham Knight brings the award-winnin...,124147.0,15901.0,NaN,102904.0,"Single-player,Steam Achievements,Full controll...","Action,Adventure","Action,Open World,Superhero,Stealth,Third Pers...",16948.0,0.570038


# Exploratory Data Analysis

In [30]:
#inserting data from 1 dataset to the next
estimated_owners = []
prices = []
last_game = ""
estimate = ""
price = 0
story_rich=[]
narrative=[]
choices_matter=[]
to_append = [False, False, False]
for i in range (len(filtered_reviews)):
    game = filtered_reviews.iloc[i,1] #sometimes its 1, sometimes its 2,,, idk why 
    if game != last_game:
        estimate = narrative_games.loc[narrative_games["AppID"]==game, "Release date"].iloc[0]
        #for some reason realease date is the header for estimated owners dont ask me
        game_tags = narrative_games.loc[narrative_games["AppID"]==game, "Tags"].iloc[0]
        to_append[0] = "Story Rich" in game_tags
        to_append[1] = "Narrative" in game_tags
        to_append[2] = "Choices Matter" in game_tags
        #same here,, required age = price
        price = narrative_games.loc[narrative_games["AppID"]==game, "Required age"].iloc[0]

    estimated_owners.append(estimate)
    story_rich.append(to_append[0])
    narrative.append(to_append[1])
    choices_matter.append(to_append[2])
    prices.append(price)
    last_game = game    


filtered_reviews["estimated_owners"] = estimated_owners 
filtered_reviews["story_rich"] = story_rich
filtered_reviews["narrative"] = narrative 
filtered_reviews["choices_matter"] = choices_matter 
filtered_reviews["price"] = prices

estimated_owners = []
prices = []
last_game = ""
estimate = ""
price = 0
story_rich=[]
narrative=[]
choices_matter=[]
filtered_reviews.head()

,app_id,app_name,review_text,review_score,review_votes,lemmatized_reviews,estimated_owners,story_rich,narrative,choices_matter,price
46177,104600,Portal 2 - The Final Hours,first of all this is no game but a virtual int...,1,1,first game virtual interactive book good one m...,200000 - 500000,True,False,False,0.79
46178,104600,Portal 2 - The Final Hours,the final hours is not a game though can't you...,1,1,final hour not game though can't imagine one s...,200000 - 500000,True,False,False,0.79
46179,104600,Portal 2 - The Final Hours,get this if you are a geek about how games are...,1,1,get geek game made |,200000 - 500000,True,False,False,0.79
46180,104600,Portal 2 - The Final Hours,it's not a game | but it's a very interesting ...,1,1,not game | interesting reading interested crea...,200000 - 500000,True,False,False,0.79
46181,104600,Portal 2 - The Final Hours,everything you would want to know about portal...,1,1,everything would want know portal right intera...,200000 - 500000,True,False,False,0.79


In [31]:
#Get reviews from top5 games in filtered reviews
all_top5_df = filtered_reviews[filtered_reviews['app_name'].isin(top5_games)]

sampled_top5_array = []

for i in range(len(top5_games)):
    temp_df=filtered_reviews[filtered_reviews["app_name"].str.contains(top5_games[i])]
    sampled_temp_df=temp_df.sample(1000)

    #sample reviews from each
    sampled_top5_array.append(sampled_temp_df)

sampled_top5_df = pd.concat(sampled_top5_array)
sampled_top5_df.head()

,app_id,app_name,review_text,review_score,review_votes,lemmatized_reviews,estimated_owners,story_rich,narrative,choices_matter,price
5122952,391540,Undertale,undertale's a great game | what you heard | it...,1,0,undertale's great game | heard | hearin' | lis...,5000000 - 10000000,True,False,True,2.49
5109922,391540,Undertale,i put a tem through college | needless to say ...,1,0,put tem college | needless say outcome expected |,5000000 - 10000000,True,False,True,2.49
5119415,391540,Undertale,this is probably my favorite rpg game yet | it...,1,1,probably favorite rpg game yet | creative funn...,5000000 - 10000000,True,False,True,2.49
5128461,391540,Undertale,i've tried and tried to like this ridiculous b...,-1,0,tried tried like ridiculous boring game | musi...,5000000 - 10000000,True,False,True,2.49
5156901,391540,Undertale,its a great and emotinal game filled with dete...,1,0,great emotinal game filled determination | pla...,5000000 - 10000000,True,False,True,2.49


In [32]:
#Get reviews from bottom5 games in filtered reviews
all_bot5_df = filtered_reviews[filtered_reviews['app_name'].isin(bot5_games)]

sampled_bot5_array = []

for i in range(len(bot5_games)):
    temp_df=filtered_reviews[filtered_reviews["app_name"].str.contains(bot5_games[i])]
    if len(temp_df) > 1000:
        sampled_temp_df=temp_df.sample(1000)

        #sample reviews from each
        sampled_bot5_array.append(sampled_temp_df)
    else:
        sampled_bot5_array.append(temp_df)

sampled_bot5_df = pd.concat(sampled_bot5_array)
sampled_bot5_df.head()

,app_id,app_name,review_text,review_score,review_votes,lemmatized_reviews,estimated_owners,story_rich,narrative,choices_matter,price
4695754,360430,Mafia III: Definitive Edition,this game in its current state | it is the ie...,-1,0,game current state | iest port pc seen since a...,1000000 - 2000000,True,False,False,5.99
4695535,360430,Mafia III: Definitive Edition,this is my first no good game | this game is n...,-1,0,first good game | game not finished | not wast...,1000000 - 2000000,True,False,False,5.99
4695347,360430,Mafia III: Definitive Edition,update: the ui issue seems to only be in the p...,-1,0,update: ui issue seems prologue | frame dip ca...,1000000 - 2000000,True,False,False,5.99
4695410,360430,Mafia III: Definitive Edition,bad pc port despite the first patch | i have a...,-1,0,bad pc port despite first patch | i-k gb ram r...,1000000 - 2000000,True,False,False,5.99
4695441,360430,Mafia III: Definitive Edition,author's note: don't judge this review by the ...,-1,0,author's note: judge review time spent pc vers...,1000000 - 2000000,True,False,False,5.99


In [33]:
all_topbot_reviews = pd.concat([all_top5_df,all_bot5_df])

sampled_reviews = pd.concat([sampled_top5_df, sampled_bot5_df])

sampled_reviews.head()

,app_id,app_name,review_text,review_score,review_votes,lemmatized_reviews,estimated_owners,story_rich,narrative,choices_matter,price
5122952,391540,Undertale,undertale's a great game | what you heard | it...,1,0,undertale's great game | heard | hearin' | lis...,5000000 - 10000000,True,False,True,2.49
5109922,391540,Undertale,i put a tem through college | needless to say ...,1,0,put tem college | needless say outcome expected |,5000000 - 10000000,True,False,True,2.49
5119415,391540,Undertale,this is probably my favorite rpg game yet | it...,1,1,probably favorite rpg game yet | creative funn...,5000000 - 10000000,True,False,True,2.49
5128461,391540,Undertale,i've tried and tried to like this ridiculous b...,-1,0,tried tried like ridiculous boring game | musi...,5000000 - 10000000,True,False,True,2.49
5156901,391540,Undertale,its a great and emotinal game filled with dete...,1,0,great emotinal game filled determination | pla...,5000000 - 10000000,True,False,True,2.49


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.style as style
from wordcloud import WordCloud

In [ ]:
def create_wordcloud(reviews, max_words=500):
    """Create a wordcloud of most common words in a set of tweets"""
    
    # Transform text for WordCloud
    reviews = reviews.dropna()
    reviews = ' '.join(reviews)
    reviews = reviews.replace(' ', ',')
    
    # Generate wordcloud image
    wc = WordCloud(background_color="white", max_words=max_words, colormap='plasma')
    wc.generate(reviews)
    plt.imshow(wc, interpolation='bilinear')
    plt.title('Review Cloud', size=30)
    plt.axis("off")

# Generate a wordcloud
create_wordcloud(filtered_reviews['review_text'])

In [ ]:
create_wordcloud(filtered_reviews['lemmatized_reviews'])

### enable gpu

In [ ]:
import torch

# Check if a CUDA-capable GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# This line checks if cuda GPU is available
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

# Sentiment Analysis

### Without Sampled Data

In [ ]:
from IPython.display import clear_output

sentiment_analysis = lambda doc: sia.polarity_scores(doc.text)
nlp_with_sentiment = spacy.load("en_core_web_sm")
Doc.set_extension("sentiment", getter=sentiment_analysis, force=True)

review_docs = nlp_with_sentiment.pipe(lemmatized_reviews, batch_size=32)
i=0
com_sent = []
pos_sent = []
neg_sent = []
neu_sent = []
for doc in review_docs:
    sentiment = doc._.sentiment
    com_sent.append(sentiment['compound'])
    pos_sent.append(sentiment['pos'])
    neg_sent.append(sentiment['neg'])
    neu_sent.append(sentiment['neu'])
    i+=1
    if(i%1000==0):
        clear_output(wait=True)
        print(str(i)+'/'+str(len(filtered_reviews)))
filtered_reviews['com_sent'] = com_sent
filtered_reviews['pos_sent'] = pos_sent
filtered_reviews['neg_sent'] = neg_sent
filtered_reviews['neu_sent'] = neu_sent

filtered_reviews.head()
filtered_reviews.to_csv("output_senti.csv")

### With Sampled Data

In [ ]:
from IPython.display import clear_output

sentiment_analysis = lambda doc: sia.polarity_scores(doc.text)
nlp_with_sentiment = spacy.load("en_core_web_sm")
Doc.set_extension("sentiment", getter=sentiment_analysis, force=True)

review_docs = nlp_with_sentiment.pipe(sampled_reviews["lemmatized_reviews"], batch_size=32)
i=0
com_sent = []
pos_sent = []
neg_sent = []
neu_sent = []
for doc in review_docs:
    sentiment = doc._.sentiment
    com_sent.append(sentiment['compound'])
    pos_sent.append(sentiment['pos'])
    neg_sent.append(sentiment['neg'])
    neu_sent.append(sentiment['neu'])
    i+=1
    if(i%1000==0):
        clear_output(wait=True)
        print(str(i)+'/'+str(len(sampled_reviews)))

sampled_reviews['com_sent'] = com_sent
sampled_reviews['pos_sent'] = pos_sent
sampled_reviews['neg_sent'] = neg_sent
sampled_reviews['neu_sent'] = neu_sent

sampled_reviews.head()
sampled_reviews.to_csv("output_senti_sampled.csv")

# Topic Modeling

References:

* https://maartengr.github.io/BERTopic/getting_started/quickstart/quickstart.html
* https://maartengr.github.io/BERTopic/getting_started/visualization/visualization.html
* https://maartengr.github.io/BERTopic/getting_started/tips_and_tricks/tips_and_tricks.html

We want to perform topic modeling on:
* reviews from the top 5 games
* reviews from the bottom 5 games
* random sample of reviews across all positive reviews
* random sample of reviews across all negative reviews

NOTE: Not recommended daw to manually stopword removal and lemmatization for BERTopic modeling. There's some tips for removing/reducing stopwords in the third ref above - moi

In [ ]:
TM_SAMPLE_N = 10000

### CRITERIA:
### - sentence form
### - sentence has at least 5 words
### - no need for stopword removal, no need for lematization

def get_review_sentences(reviews_list, min_word_count=5):
    sentences = []
    for i, review in enumerate(reviews_list):
        for sentence in (str(review)).split(" | "):
            if (len(sentence.split()) >= min_word_count):
                sentences.append(("").join(sentence))

    print("Sentences retrieved: " + str(len(sentences)))
    return sentences

In [ ]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
# from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer


TM_DEFAULT_MIN_TOPIC_SIZE = 200
RANDOM_SEED = 230995 

sentence_model = SentenceTransformer("all-MiniLM-L6-v2")

vectorizer_model = CountVectorizer(stop_words="english")

In [ ]:
def perform_topic_modeling(data, min_cluster_size=TM_DEFAULT_MIN_TOPIC_SIZE, data_title=""):
    MIN_TOPIC_COUNT_WANTED = 5
    hdbscan_model = HDBSCAN(min_cluster_size=min_cluster_size, 
                        metric='euclidean',
                        cluster_selection_method='eom', 
                        prediction_data=True
                    )
    
    topic_model = BERTopic(hdbscan_model=hdbscan_model, vectorizer_model=vectorizer_model)
    embeddings = sentence_model.encode(data, show_progress_bar=True, batch_size=32)
    topic_model.fit(data)

    ## IT WILL ERROR IF THERE ARENT ENOUGH TOPICS
    ## so make sure to check
    topic_info = topic_model.get_topic_info()
    topics_count = len(set(topic_info["Topic"])) - 1 # - 1 since there's also a category for outlier topics
    while topics_count < MIN_TOPIC_COUNT_WANTED:
        print("Not enough topics found when min_cluster_size=" + str(hdbscan_model.min_cluster_size) + ". Reducing min cluster size by 5 then rerunning.")
        topic_model.hdbscan_model.set_params(min_cluster_size=(hdbscan_model.min_cluster_size - 5))
        topic_model.fit(data)
        topic_info = topic_model.get_topic_info()
        topics_count = len(set(topic_info["Topic"])) - 1

    data_title = data_title + " | " if (data_title != "") else ""
    vistopics = topic_model.visualize_topics(title=(data_title + "Intertopic Distance Map"))
    visdocus = topic_model.visualize_documents(data, embeddings=embeddings, title=(data_title + "Documents and Topics"))

    return topic_info, vistopics, visdocus


### Positive reviews

In [ ]:
willrecco_df = filtered_reviews[filtered_reviews.review_score==1]
willrecco_reviews = (willrecco_df["review_text"].sample(n=TM_SAMPLE_N, random_state = RANDOM_SEED)).tolist()
willrecco_sentences = get_review_sentences(willrecco_reviews)

In [ ]:
willrecco_topicinfo, willrecco_topicsvis, willrecco_docusvis = perform_topic_modeling(willrecco_sentences, 
                                                                                      data_title=(str(TM_SAMPLE_N) + " Would-Recommend Reviews")
                                                                                      )
willrecco_topicsvis.show()
willrecco_docusvis.show()

### Negative reviews

In [ ]:
notrecco_df = filtered_reviews[filtered_reviews.review_score==-1]
notrecco_reviews = (notrecco_df["review_text"].sample(n=TM_SAMPLE_N, random_state = RANDOM_SEED)).tolist()
notrecco_sentences = get_review_sentences(notrecco_reviews)

In [ ]:
notrecco_topicinfo, notrecco_topicsvis, notrecco_docusvis = perform_topic_modeling(notrecco_sentences, 
                                                                                   data_title=(str(TM_SAMPLE_N) + " Would-Not-Recommend Reviews")
                                                                                   )
notrecco_topicsvis.show()
notrecco_docusvis.show()

### All Positive Reviews from Top 5 Games

In [ ]:
# print(len((all_top5_df[all_top5_df.review_score==1])["review_text"].to_list()))
all_top5_reviews = (all_top5_df[all_top5_df.review_score==1])["review_text"].to_list()
all_top5_sentences = get_review_sentences(all_top5_reviews)

In [ ]:
all_top5_topicinfo, all_top5_topicvis, all_top5_docusvis = perform_topic_modeling(all_top5_sentences, 
                                                                                  min_cluster_size=1000,
                                                                                  data_title="Positive Reviews from Top 5 Games")
all_top5_topicvis.show()
all_top5_docusvis.show()

### All Negative Reviews from Bottom 5 Games

In [ ]:
print(len((all_bot5_df[all_bot5_df.review_score==-1])["review_text"].to_list()))
all_bot5_reviews = (all_bot5_df[all_bot5_df.review_score==-1])["review_text"].to_list()
all_bot5_sentences = get_review_sentences(all_bot5_reviews)

In [ ]:
## this code snippet was initially run with default min cluster size (i.e. min cluster size = 200)
## but topics found were less than 5 until min cluster size was reduced to ___

all_bot5_topicinfo, all_bot5_topicvis, all_bot5_docusvis = perform_topic_modeling(all_bot5_sentences, 
                                                                                  min_cluster_size = 300,
                                                                                  data_title="Negative Reviews from Bottom 5 Games")
all_bot5_topicvis.show()
all_bot5_docusvis.show()

### Topic Modeling Per Game

In [ ]:
## BUG: Portal 2 appears in 'top5_games twice
# temp workaround is this cell

top5_games = list(set(top5_games))
print(top5_games)

In [ ]:
for game in top5_games:
    game_reviews = (all_top5_df[all_top5_df.app_name==game])["review_text"]
    print(game)
    game_sentences = get_review_sentences(game_reviews)



    topic_info, topicvis, docusvis = perform_topic_modeling(game_sentences, 
                                                            min_cluster_size=int(len(game_sentences) * 0.01), 
                                                            data_title=game)

    topicvis.show()
    docusvis.show()

In [ ]:
## BUG: Portal 2 appears in 'top5_games twice
# temp workaround is this cell

bot5_games = list(set(bot5_games))
print(bot5_games)

In [ ]:
for game in bot5_games:
    game_reviews = (all_bot5_df[all_bot5_df.app_name==game])["review_text"]
    print(game)
    # print("---> " + str(len(game_reviews)))
    game_sentences = get_review_sentences(game_reviews)


    
    topic_info, topicvis, docusvis = perform_topic_modeling(game_sentences, 
                                                            min_cluster_size=int(len(game_sentences) * 0.001), 
                                                            data_title=game)

    topicvis.show()
    docusvis.show()